# LungLens — training on TBX11K
TB screening from chest X-rays: **healthy / sick but not TB / TB**.

**Runtime → Run all.** About 35 minutes on an L4 GPU. Approve Google Drive access when asked.
Everything is saved to *Google Drive → Deep Learning Projects → LungLens → outputs*, one model at a time,
so a disconnect loses at most the model that was training (re-running skips finished models).

Three models, same data, same augmentation:
1. `densenet121_cxr` — DenseNet-121 pretrained on chest X-rays (TorchXRayVision)
2. `densenet121_imagenet` — the same network pretrained on ImageNet photos
3. `cnn_scratch` — a 5-block CNN from random weights (baseline)

In [3]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/Deep Learning Projects/LungLens/outputs'
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT)

GPU 0: NVIDIA L4 (UUID: GPU-c75cfef4-2e04-9829-9eee-312056b8e782)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saving to /content/drive/MyDrive/Deep Learning Projects/LungLens/outputs


In [4]:
# code (the same files as the GitHub repo)
!rm -rf /content/lunglens && git clone -q https://github.com/zyna-b/lunglens /content/lunglens
%cd /content/lunglens
!pip install -q torchxrayvision
!ls -R /content/lunglens | head -20

/content/lunglens
/content/lunglens:
data
lunglens
requirements.txt
train.py

/content/lunglens/data:
splits.csv

/content/lunglens/lunglens:
augment.py
data.py
metrics.py
models.py


In [5]:
%%bash
# TBX11K (CC BY 4.0) straight from Kaggle, ~3 GB
set -e
if [ ! -f /content/tbx11k/data.csv ]; then
  curl -sL -o /content/tbx.zip https://www.kaggle.com/api/v1/datasets/download/vbookshelf/tbx11k-simplified
  mkdir -p /content/tbx && unzip -q /content/tbx.zip -d /content/tbx && rm /content/tbx.zip
  mv /content/tbx/tbx11k-simplified /content/tbx11k
fi
echo "images: $(ls /content/tbx11k/images | wc -l)"

images: 8400


## 1. DenseNet-121, chest-X-ray pretrained (main model)

In [6]:
!python train.py --data /content/tbx11k --model densenet121_cxr --out "{OUT}" --skip-done 2>&1 | grep --line-buffered -v -i warn | tee -a "{OUT}/train_log.txt"

{'train': 5939, 'val': 660, 'test': 1800} | device: cuda NVIDIA L4
cached 8399 images at 384px in 23s
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
ep  1/20  loss 0.9898  val acc 0.9091  bal 0.8089  tbAUC 0.9321  55s  *best
ep  2/20  loss 0.3998  val acc 0.9394  bal 0.9422  tbAUC 0.9828  51s  *best
ep  3/20  loss 0.2312  val acc 0.9288  bal 0.9211  tbAUC 0.9867  51s
ep  4/20  loss 0.1757  val acc 0.9242  bal 0.9444  tbAUC 0.9939  51s  *best
ep  5/20  loss 0.1504  val acc 0.9455  bal 0.9511  tbAUC 0.9935  51s  *best
ep  6/20  loss 0.1258  val acc 0.9773  bal 0.9656  tbAUC 0.9964  51s  *best
ep  7/20  loss 0.1122  val acc 0.9788  bal 0.9711  tbAUC 0.9972  51s 

## 2. DenseNet-121, ImageNet pretrained

In [7]:
!python train.py --data /content/tbx11k --model densenet121_imagenet --out "{OUT}" --skip-done 2>&1 | grep --line-buffered -v -i warn | tee -a "{OUT}/train_log.txt"

{'train': 5939, 'val': 660, 'test': 1800} | device: cuda NVIDIA L4
cached 8399 images at 384px in 23s
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 241MB/s]
ep  1/20  loss 0.4776  val acc 0.7439  bal 0.8122  tbAUC 0.9919  53s  *best
ep  2/20  loss 0.1990  val acc 0.9394  bal 0.9467  tbAUC 0.9916  51s  *best
ep  3/20  loss 0.1137  val acc 0.9606  bal 0.9667  tbAUC 0.9973  51s  *best
ep  4/20  loss 0.0915  val acc 0.9621  bal 0.9633  tbAUC 0.9981  51s
ep  5/20  loss 0.0916  val acc 0.8955  bal 0.9189  tbAUC 0.9952  51s
ep  6/20  loss 0.0553  val acc 0.9818  bal 0.9867  tbAUC 0.9991  51s  *best
ep  7/20  loss 0.0646  val acc 0.9621  bal 0.9678  tbAUC 0.9972  51s
ep  8/20  loss 0.0353  val acc 0.9742  bal 0.9589  tbAUC 0.9949  51s
ep  9/20  loss 0.0359  val acc 0.9894  bal 0.9922  tbAUC 0.9995  51s  *best
ep 10/20  loss 0.0258  val acc 0.9879  bal 0.9911  

## 3. CNN from scratch (baseline)

In [8]:
!python train.py --data /content/tbx11k --model cnn_scratch --out "{OUT}" --skip-done 2>&1 | grep --line-buffered -v -i warn | tee -a "{OUT}/train_log.txt"

{'train': 5939, 'val': 660, 'test': 1800} | device: cuda NVIDIA L4
cached 8399 images at 384px in 24s
ep  1/40  loss 0.7443  val acc 0.4909  bal 0.3600  tbAUC 0.4964  27s  *best
ep  2/40  loss 0.6160  val acc 0.8879  bal 0.7844  tbAUC 0.8760  26s  *best
ep  3/40  loss 0.5443  val acc 0.6485  bal 0.5067  tbAUC 0.6945  26s
ep  4/40  loss 0.4893  val acc 0.4818  bal 0.4511  tbAUC 0.7596  26s
ep  5/40  loss 0.4825  val acc 0.9152  bal 0.8400  tbAUC 0.9355  26s  *best
ep  6/40  loss 0.4208  val acc 0.8500  bal 0.7478  tbAUC 0.8763  26s
ep  7/40  loss 0.4278  val acc 0.5894  bal 0.6233  tbAUC 0.8317  26s
ep  8/40  loss 0.3805  val acc 0.8697  bal 0.8511  tbAUC 0.9363  26s  *best
ep  9/40  loss 0.3647  val acc 0.9106  bal 0.8767  tbAUC 0.9530  26s  *best
ep 10/40  loss 0.3390  val acc 0.4379  bal 0.5656  tbAUC 0.8510  26s
ep 11/40  loss 0.3363  val acc 0.5591  bal 0.6411  tbAUC 0.8644  26s
ep 12/40  loss 0.2964  val acc 0.9121  bal 0.8067  tbAUC 0.9574  26s
ep 13/40  loss 0.2863  val acc 0.80

## Results on the 1,800 held-out test X-rays

In [9]:
import json, glob
print(f"{'model':22s} {'acc':>6s} {'bal.acc':>7s} {'TB AUC':>7s} {'sens':>6s} {'spec':>6s} {'pointing':>8s}")
for p in sorted(glob.glob(f'{OUT}/*/metrics.json')):
    m = json.load(open(p)); t = m['test']; s = t['tb_vs_rest']
    print(f"{m['model']:22s} {t['accuracy']:6.4f} {t['balanced_accuracy']:7.4f} {s['auc']:7.4f} {s['sensitivity']:6.3f} {s['specificity']:6.3f} {m['pointing_game']['hit_rate']:8.3f}")

model                     acc bal.acc  TB AUC   sens   spec pointing
cnn_scratch            0.9694  0.9696  0.9952  0.840  0.997    0.340
densenet121_cxr        0.9833  0.9700  0.9965  0.835  0.998    0.575
densenet121_imagenet   0.9922  0.9842  0.9996  0.850  1.000    0.360


In [10]:
# one zip for the project's GitHub repo
!cd "{OUT}/.." && rm -f lunglens_outputs.zip && zip -qr lunglens_outputs.zip outputs && ls -la "{OUT}/.."
print('DONE')

total 61380
drwx------ 3 root root     4096 Oct 27 22:10 .
drwx------ 4 root root     4096 Oct 27 20:56 ..
-rw------- 1 root root 62825365 Oct 27 22:10 lunglens_outputs.zip
-rw------- 1 root root    14675 Oct 27 22:09 LungLens_train.ipynb
drwx------ 5 root root     4096 Oct 27 22:10 outputs
DONE


In [ ]:
# Final step: re-score all three models with the final screening rule, save this notebook, zip the outputs
!git -C /content/lunglens pull -q
%run scripts/colab_finish.py